# Load Data

In [2]:
TABLE = "price_features"
CALENDAR = "no_calendar"

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\maxan\OneDrive\Desktop\0. Personal Projects\market-intelligence-pipeline")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    calendar_df["date"] = pd.to_datetime(calendar_df["date"])

    calendar_df = calendar_df.sort_values(["ticker", "date"])

    print(f"[DONE] Prepared and sorted table")

    return calendar_df

In [5]:
df = pull_calendar_table(
    table_name = TABLE,
    calendar = CALENDAR
)

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[DONE] Prepared and sorted table


In [6]:
df.columns

Index(['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
       'daily_return_no_calendar', 'log_return_no_calendar',
       'cumulative_returns_no_calendar', 'rolling_7d_return_no_calendar',
       'rolling_30d_return_no_calendar', 'lag_1_return_no_calendar',
       'lag_5_return_no_calendar', 'moving_avg_20_no_calendar',
       'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar',
       'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar',
       'drawdown_no_calendar', 'target_next_day_return_no_calendar',
       'target_direction_no_calendar'],
      dtype='str')

In [7]:
table_count = 0
graph_count = 0
matrix_count = 0

# Feature Correlation with Target Next Day Return

In [34]:
df_1 = df.copy()

columns_to_keep = [
    column
    for column in df_1.columns
    if column == "ticker"
    or column.endswith(f"_{CALENDAR}")
]

df_1 = df_1[columns_to_keep]

df_1 = df_1.drop(columns=f"target_direction_{CALENDAR}", errors="ignore")

target = f"target_next_day_return_{CALENDAR}"

pearson = (
    df_1.corr(method="pearson", numeric_only=True)[target]
        .drop(target)
)

spearman = (
    df_1.corr(method="spearman", numeric_only = True)[target]
        .drop(target)
)

summary = (
    pd.DataFrame({
        "pearson":pearson,
        "spearman":spearman
    })
    .assign(
        max_absolute_correlation=lambda x:
            x[["pearson", "spearman"]].abs().max(axis=1)
    )
    .sort_values(
        "max_absolute_correlation",
        ascending=False
    )
)

table_count += 1
print(f"Table {table_count}: Global Pearson and Spearman Feature-Target Correlations")
display(summary.style.format(precision=3))

,pearson,spearman,max_absolute_correlation
rolling_30d_volatility_no_calendar,0.095,0.022,0.095
cumulative_returns_no_calendar,0.070,0.019,0.070
rolling_30d_return_no_calendar,0.057,0.016,0.057
moving_avg_20_no_calendar,0.036,0.009,0.036
price_vs_ma20_no_calendar,0.032,0.000,0.032
lag_5_return_no_calendar,-0.030,-0.014,0.030
moving_avg_50_no_calendar,0.029,0.010,0.029
drawdown_no_calendar,0.004,0.019,0.019
daily_return_no_calendar,-0.014,-0.018,0.018
log_return_no_calendar,-0.016,-0.018,0.018


Pearson correlation measures linear relationship strength while Spearman correlation measures monotonic relationship strength meaning the tendency for two variables to go up but not necessarily in a linear pattern.

Globally, no features have meaningful standalone relationships with next day returns.

Small signals that high recent volatility is associated with higher returns, as well as slight signals of positive momentum with positive recent returns linked with positive returns.

In [38]:
target = f"target_next_day_return_{CALENDAR}"
ticker_column = "ticker"

pearson_by_ticker = pd.DataFrame({
    ticker_value: (
        group
        .corr(method="pearson", numeric_only=True)[target]
        .drop(target)
    )
    for ticker_value, group in df_1.groupby(ticker_column)
})

pearson_by_ticker = (
    pearson_by_ticker
    .rename_axis("variable")
    .reset_index()
    .assign(
        max_absolute_correlation=lambda x:
            x[pearson_by_ticker.columns].abs().max(axis=1),

        max_correlation_ticker=lambda x:
            x[pearson_by_ticker.columns].abs().idxmax(axis=1)
    )
    .sort_values(
        "max_absolute_correlation",
        ascending=False
    )
)

table_count += 1
print(f"Table {table_count}: Feature-Target Pearson Correlation by Ticker")

display(
    pearson_by_ticker.style
    .hide(axis="index")
    .format(precision=3)
)

variable,GLD,MU,NKE,RPI.L,SNDK,SPY,TLT,max_absolute_correlation,max_correlation_ticker
daily_return_no_calendar,0.004,-0.043,-0.049,0.080,-0.041,-0.134,-0.016,0.134,SPY
log_return_no_calendar,0.004,-0.042,-0.050,0.076,-0.037,-0.134,-0.016,0.134,SPY
lag_5_return_no_calendar,-0.038,-0.050,-0.038,0.004,-0.059,-0.101,0.016,0.101,SPY
lag_1_return_no_calendar,-0.036,0.030,0.020,-0.042,-0.093,0.081,-0.094,0.094,TLT
rolling_30d_volatility_no_calendar,-0.003,0.060,0.050,0.089,0.048,0.038,0.004,0.089,RPI.L
moving_avg_50_no_calendar,0.004,0.082,-0.020,-0.088,0.022,-0.001,-0.022,0.088,RPI.L
relative_volume_no_calendar,-0.045,-0.005,-0.034,0.032,0.079,-0.021,-0.003,0.079,SNDK
moving_avg_20_no_calendar,0.007,0.076,-0.021,-0.050,0.040,-0.002,-0.019,0.076,MU
cumulative_returns_no_calendar,0.003,0.068,-0.026,-0.042,0.019,-0.007,-0.025,0.068,MU
drawdown_no_calendar,0.009,-0.006,0.018,-0.012,0.041,-0.057,0.001,0.057,SPY


Across tickers, still generally weak correlations, although stronger relationships compared to global analysis.

SPY shows strongest very short term reversal trends with recent positive returns correlating slightly with negative target returns.

In [40]:
spearman_by_ticker = pd.DataFrame({
    ticker_value: (
        group
        .corr(method="spearman", numeric_only=True)[target]
        .drop(target)
    )
    for ticker_value, group in df_1.groupby(ticker_column)
})

spearman_by_ticker = (
    spearman_by_ticker
    .rename_axis("variable")
    .reset_index()
    .assign(
        max_absolute_correlation=lambda x:
            x[spearman_by_ticker.columns].abs().max(axis=1),

        max_correlation_ticker=lambda x:
            x[spearman_by_ticker.columns].abs().idxmax(axis=1)
    )
    .sort_values(
        "max_absolute_correlation",
        ascending=False
    )
)

table_count += 1
print(f"Table {table_count}: Feature-Target Spearman Correlation by Ticker")

display(
    spearman_by_ticker.style
    .hide(axis="index")
    .format(precision=3)
)

variable,GLD,MU,NKE,RPI.L,SNDK,SPY,TLT,max_absolute_correlation,max_correlation_ticker
lag_1_return_no_calendar,-0.017,0.011,0.012,0.020,-0.113,-0.014,-0.044,0.113,SNDK
relative_volume_no_calendar,-0.001,-0.004,-0.030,-0.012,0.088,0.005,-0.011,0.088,SNDK
moving_avg_50_no_calendar,0.032,0.027,-0.016,-0.039,0.063,-0.011,-0.031,0.063,SNDK
moving_avg_20_no_calendar,0.035,0.018,-0.019,-0.021,0.063,-0.013,-0.031,0.063,SNDK
rolling_30d_volatility_no_calendar,0.007,0.027,0.026,0.044,0.046,0.034,-0.032,0.046,SNDK
rolling_30d_return_no_calendar,0.046,0.000,0.001,0.004,0.027,-0.030,-0.000,0.046,GLD
daily_return_no_calendar,-0.022,-0.027,-0.033,0.021,-0.046,-0.026,-0.028,0.046,SNDK
log_return_no_calendar,-0.022,-0.027,-0.033,0.021,-0.046,-0.026,-0.028,0.046,SNDK
cumulative_returns_no_calendar,0.034,0.012,-0.024,-0.006,0.045,-0.019,-0.037,0.045,SNDK
drawdown_no_calendar,0.038,-0.010,0.024,0.036,0.021,-0.042,-0.015,0.042,SPY


Still weak signals but SNDK generally dominate showing the strongest spearman correlation signals. Shows momentum, trend, and mean reversion signals as moving average returns has positive correlations with target returns, positive relative volumne returns, but negative correlation with lag returns indicating that when previous recent days returns are high then the next day returns after tends to be lower.

Pearson evaluates linear relationship between variables making it more sensitive to outliers whiel Spearman compares monotonic relationship between ranks making it more resistant to outliers.

Spearman correlations between the engineered features and next-day returns are overwhelmingly negligible, with no consistent relationship across assets. The strongest result is a weak negative association between SNDK’s lagged return and its next-day return (ρ = −0.113), suggesting limited mean-reversion behaviour, while SNDK relative volume has a weak positive association (ρ = 0.088). Current daily and log returns are slightly negatively related to next-day returns for six of seven assets, but all coefficients remain below |0.05|, providing only minimal evidence of short-term reversal. Rolling returns, moving-average position, volatility, drawdown and cumulative-return features likewise show small and frequently mixed relationships. Comparison with Pearson correlations should determine whether these patterns are broadly linear or whether they are influenced by outliers, but the overall results indicate that individual technical features offer little standalone predictive power for next-day returns.

# Features vs Target Direction

In [44]:
df_2 = df.copy()

columns_to_keep = [
    column
    for column in df_2.columns
    if column == "ticker" or
    column.endswith(f"_{CALENDAR}")
]

df_2 = df_2[columns_to_keep]

df_2 = df_2.drop(columns=[f"target_next_day_return_{CALENDAR}"], errors="ignore")

target = f"target_direction_{CALENDAR}"

feature_avg = (
    df_2.groupby(target)
    .mean(numeric_only=True)
    .T
    .rename(columns={
        0: "down_avg",
        1: "up_avg"
    })
    .assign(
        difference = lambda x:
        x["up_avg"] - x["down_avg"]
    )
    .sort_values("difference", ascending=False)
)

table_count += 1
print(f"Table {table_count}: Global Comparison of Feature Means on Up vs Down Days")

display(
    feature_avg.style
        .format(precision=4)
)

feature_med = (
    df_2.groupby(target)
    .median(numeric_only=True)
    .T
    .rename(columns={
        0: "down_avg",
        1: "up_avg"
    })
    .assign(
        difference = lambda x:
        x["up_avg"] - x["down_avg"]
    )
    .sort_values("difference", ascending=False)
)

table_count += 1
print(f"Table {table_count}: Global Comparison of Feature Medians on Up vs Down Days")

display(
    feature_med.style
        .format(precision=4)
)

Table 1: Global Comparison of Feature Means on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
moving_avg_50_no_calendar,189.0548,196.5850,7.5302
moving_avg_20_no_calendar,192.8172,200.2789,7.4617
cumulative_returns_no_calendar,0.8937,0.9786,0.0849
drawdown_no_calendar,-0.1817,-0.1693,0.0124
rolling_30d_return_no_calendar,0.0354,0.0377,0.0023
price_vs_ma20_no_calendar,0.0087,0.0088,0.0001
rolling_30d_volatility_no_calendar,0.0181,0.0182,0.0001
lag_1_return_no_calendar,0.0012,0.0011,-0.0001
rolling_7d_return_no_calendar,0.0081,0.0077,-0.0004
lag_5_return_no_calendar,0.0014,0.0009,-0.0005


Table 2: Global Comparison of Feature Medians on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
moving_avg_50_no_calendar,119.8356,122.9978,3.1622
moving_avg_20_no_calendar,119.6234,122.7809,3.1575
drawdown_no_calendar,-0.1151,-0.0986,0.0166
cumulative_returns_no_calendar,0.3499,0.3526,0.0027
rolling_30d_return_no_calendar,0.0110,0.0126,0.0016
lag_5_return_no_calendar,0.0006,0.0006,-0.0000
price_vs_ma20_no_calendar,0.0049,0.0047,-0.0002
lag_1_return_no_calendar,0.0007,0.0005,-0.0002
rolling_30d_volatility_no_calendar,0.0129,0.0124,-0.0005
log_return_no_calendar,0.0009,0.0004,-0.0005


Globally, there is generally no difference in feature values when comparing up and down days. Moving averages of adjusted close may seem high but remember these are calculated in the real terms of the assets and are not very reliable since assets have varying prices.

In [46]:
target = f"target_direction_{CALENDAR}"
ticker="ticker"

for asset, group in df_2.groupby(ticker):
    feature_ticker_mean = (
    group.groupby(target)
    .mean(numeric_only=True)
    .T
    .rename(columns={
        0: "down_avg",
        1: "up_avg"
    })
    .assign(
        difference = lambda x:
        x["up_avg"] - x["down_avg"]
    )
    .sort_values("difference", ascending=False)
)

    table_count += 1
    print(f"Table {table_count}: {asset} Comparison of Feature Means on Up vs Down Days")

    display(
        feature_ticker_mean.style
            .format(precision=4)
    )

Table 4: GLD Comparison of Feature Means on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
moving_avg_20_no_calendar,196.4514,198.9581,2.5067
moving_avg_50_no_calendar,195.8377,197.5942,1.7565
cumulative_returns_no_calendar,0.5744,0.5936,0.0192
drawdown_no_calendar,-0.0709,-0.0669,0.0039
rolling_30d_return_no_calendar,0.0161,0.0194,0.0033
lag_5_return_no_calendar,0.0004,0.0007,0.0003
rolling_30d_volatility_no_calendar,0.0097,0.0096,-0.0002
rolling_7d_return_no_calendar,0.0042,0.0038,-0.0004
price_vs_ma20_no_calendar,0.0053,0.0049,-0.0004
log_return_no_calendar,0.0008,0.0003,-0.0005


Table 5: MU Comparison of Feature Means on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
moving_avg_50_no_calendar,90.3471,94.9290,4.5819
moving_avg_20_no_calendar,95.4445,100.0032,4.5587
cumulative_returns_no_calendar,1.3387,1.4452,0.1065
relative_volume_no_calendar,1.0008,1.0211,0.0202
lag_1_return_no_calendar,0.0015,0.0026,0.0011
rolling_30d_volatility_no_calendar,0.0305,0.0308,0.0003
drawdown_no_calendar,-0.2281,-0.2288,-0.0008
lag_5_return_no_calendar,0.0030,0.0013,-0.0018
daily_return_no_calendar,0.0033,0.0010,-0.0023
log_return_no_calendar,0.0028,0.0004,-0.0024


Table 6: NKE Comparison of Feature Means on Up vs Down Days


...

target_direction_no_calendar,down_avg,up_avg,difference
relative_volume_no_calendar,0.9688,1.0947,0.1259
cumulative_returns_no_calendar,10.4202,10.4419,0.0217
price_vs_ma20_no_calendar,0.1106,0.1245,0.0140
log_return_no_calendar,0.0114,0.0121,0.0008
daily_return_no_calendar,0.0135,0.0142,0.0007
rolling_7d_return_no_calendar,0.0970,0.0974,0.0004
rolling_30d_volatility_no_calendar,0.0590,0.0587,-0.0004
drawdown_no_calendar,-0.1307,-0.1332,-0.0025
rolling_30d_return_no_calendar,0.5195,0.5089,-0.0106
lag_5_return_no_calendar,0.0205,0.0095,-0.0110


Table 9: SPY Comparison of Feature Means on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
moving_avg_20_no_calendar,408.2787,409.3633,1.0846
moving_avg_50_no_calendar,407.2549,408.0588,0.8039
drawdown_no_calendar,-0.0512,-0.0482,0.0030
cumulative_returns_no_calendar,0.7353,0.7370,0.0017
lag_1_return_no_calendar,0.0002,0.0010,0.0008
rolling_30d_return_no_calendar,0.0176,0.0180,0.0005
rolling_30d_volatility_no_calendar,0.0104,0.0103,-0.0001
price_vs_ma20_no_calendar,0.0052,0.0050,-0.0002
lag_5_return_no_calendar,0.0008,0.0004,-0.0004
rolling_7d_return_no_calendar,0.0044,0.0039,-0.0005


Table 10: TLT Comparison of Feature Means on Up vs Down Days


target_direction_no_calendar,down_avg,up_avg,difference
drawdown_no_calendar,-0.2299,-0.2282,0.0016
lag_5_return_no_calendar,0.0000,-0.0001,-0.0001
rolling_30d_volatility_no_calendar,0.0091,0.0090,-0.0001
rolling_30d_return_no_calendar,-0.0004,-0.0007,-0.0004
daily_return_no_calendar,0.0003,-0.0003,-0.0007
log_return_no_calendar,0.0003,-0.0004,-0.0007
lag_1_return_no_calendar,0.0005,-0.0005,-0.0010
price_vs_ma20_no_calendar,0.0004,-0.0013,-0.0017
rolling_7d_return_no_calendar,0.0011,-0.0013,-0.0024
cumulative_returns_no_calendar,0.0298,0.0217,-0.0080


Genrally, feature separation between up and down days is generally small and inconsistent across assets.

There is no single feature that bahves consistently across assets supporting conclusions drawn when looking at comaprisons globally.

From the limited signals we do have we may be able to interpret the following:

GLD may exhibit weak short term reversal within a slightly positive longer term momentum environment, but the diffrences alone are likely to exhibit predictive power.

MU provides some evidence that stronger multi-day and monthly performance is followed by short term reversal, although the immediately lagged return and volume show a modest positive association with up days.

NKE displays possible one day mean reversion, with up days tending to follow a negative return. Higher relative volume appears mroe associated with subsequent down days.

RPI.L shows the clearest medium term momentum patterns. Up days generally occur during periods of strong recent momentum, higher volume and slightly greater volatility. However, current and immediately precending returns were higher before down days suggesting short term reversal within a broader momentum regime.

SNDK contains the largest mean differences but shoudl be interpreted carefully due to its shorter history. Relative volume was seen higher during up days, as well as price relative to moving average. LAgged returns indicate substatial reversal with higher lagged returns on down days comapred to up days.

SPY diaplys very little differences and separation. There is a weak suggestion of current day reversal, but most features appear unlikely to distinguish next day direction independently.

TLT shows reasonably consistent direction of short term reversal, but the absolute differences are small and probably insufficient for reliable next day classification.

These results provide some evidence of weak short-term mean reversion, especially in the current daily return. RPI.L shows a combination of medium-term momentum and immediate reversal, while SNDK shows potentially meaningful relationships with relative volume and lagged returns. TLT also has a relatively consistent, though small, reversal pattern.

Nevertheless, the differences are mostly small and vary substantially across assets. There is no universal feature that clearly separates next-day up and down movements. The tables therefore support the conclusion that next-day direction is highly noisy and difficult to predict using individual historical price and volume features alone.

They do not mean that the features are completely useless. Some may add value jointly, through nonlinear relationships or within asset-specific models, but the mean comparisons do not reveal a strong standalone predictor.